# pyrepl

> A Python prompt you own, and an agent in the layer above it.

One terminal holds two things. The Python prompt's namespace belongs to whoever is typing;
the agent reads that namespace and builds in a layer of its own, and cannot rebind a name it
did not create. That guarantee is not this module's: Dhrishti serves the kernel's namespace
and splits its API in two, and everything here does is point the agent at the half that
cannot mutate anything.

In [ ]:
#| default_exp pyrepl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import asyncio, codeop, json, os, queue, shutil, sys, tempfile, threading, urllib.parse, urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from rich.text import Text
from ramabana.core import agent_err
from ramabana.tools import LocalHost, WRITE_TOOLS
from ramabana.agent import Agent, Approvals
from ramabana.cli import Ui, GRUVBOX, HELP

## Outputs

A kernel reports its results as a stream of messages; a notebook stores them as a list of
dicts. `ExecOutcome` is one request in the notebook's shape, which is what lets the same
outputs go to the terminal, to `log_cell` and to a test without a second representation.

In [ ]:
#| export
@dataclass
class ExecOutcome:
    "One kernel request in nbformat's output shape."
    ok: bool = True
    outputs: list = field(default_factory=list)
    execution_count: int | None = None
    error: str | None = None

def output_text(outputs):
    "Flatten notebook outputs for tests, logs and plain terminal fallbacks."
    parts = []
    for out in outputs:
        kind = out.get('output_type')
        if kind == 'stream': parts.append(_text(out.get('text')))
        elif kind in ('execute_result', 'display_data'):
            data = out.get('data') or {}
            parts.append(_text(data.get('text/plain') or data.get('text/markdown')))
        elif kind == 'error':
            trace = out.get('traceback') or []
            parts.append('\n'.join(trace) if trace else f"{out.get('ename')}: {out.get('evalue')}")
    return '\n'.join(p.rstrip('\n') for p in parts if p is not None)

def _text(value):
    return ''.join(value) if isinstance(value, list) else str(value or '')

In [ ]:
outs = [{'output_type': 'stream', 'text': 'hello\n'},
        {'output_type': 'execute_result', 'data': {'text/plain': '42'}},
        {'output_type': 'error', 'ename': 'ValueError', 'evalue': 'bad', 'traceback': []}]
test_eq(output_text(outs), 'hello\n42\nValueError: bad')
# A stream arrives split across messages, so the list form has to flatten rather than repr.
test_eq(output_text([{'output_type': 'stream', 'text': ['a', 'b']}]), 'ab')
# An error with a traceback prefers it: the ename alone loses where it happened.
test_eq(output_text([{'output_type': 'error', 'ename': 'E', 'evalue': 'v',
                      'traceback': ['line one', 'line two']}]), 'line one\nline two')
test_eq(output_text([]), '')